# WP1A — Benchmark reproduzível de métodos clássicos de ML no Tennessee Eastman Process
**Disciplina MEI0028/MEI0015 – Modelagem e Simulação · PUC Goiás · 2026 · Prof. Clarimar J. Coelho**

Base: Rieth et al. (2017), Harvard Dataverse, DOI 10.7910/DVN/6C3JR1 (domínio público) — 21 classes, 52 variáveis, 500 execuções/classe.
Protocolo: divisão **por execução (run)**; hiperparâmetros só na validação; teste lido uma única vez; 5 sementes; F1 macro + MCC; custo computacional medido no mesmo hardware.

Estes notebooks reproduzem exatamente o código executado. Cada célula `%%writefile` grava o script numerado; a célula seguinte o executa.

## Notebook 1 de 3 — Dados, auditoria, divisão por execução e análise exploratória

### 0. Ambiente
Cria a estrutura de pastas do repositório (§20 do WP1A) e instala dependências.

In [ ]:
import os; os.makedirs("/content/ProjetoA_WP1A/src",exist_ok=True); os.chdir("/content/ProjetoA_WP1A")
os.environ["WP1A_ROOT"]="/content/ProjetoA_WP1A"; os.environ["NJOBS"]="2"
!pip -q install pyreadr pyarrow tabulate 2>/dev/null; print("ambiente pronto")

### 1. Download com verificação de integridade
Os 4 arquivos `.RData` (1,4 GB) vêm do Harvard Dataverse. O servidor recusa o *user-agent* padrão do Python (HTTP 403), por isso usa-se `curl`. O **MD5 de cada arquivo é conferido** contra o publicado na API do repositório: qualquer byte alterado no download invalidaria o arquivo.

In [ ]:
%%writefile /content/ProjetoA_WP1A/src/01_download.py
"""01_download.py — baixa os 4 arquivos do dataset Rieth et al. (2017) e verifica MD5.
Fonte: Harvard Dataverse, DOI 10.7910/DVN/6C3JR1. Metadados via API em 2026-09-06.
Usa curl com user-agent de navegador (o Dataverse retorna 403 ao UA padrão do Python)."""
import hashlib, json, os, subprocess, sys, time
HERE=os.path.dirname(os.path.abspath(__file__)); RAW=os.path.join(HERE,"..","data","raw")
META=os.path.join(HERE,"..","results","metadata")
UA="Mozilla/5.0 (Macintosh; Intel Mac OS X 14_0) AppleWebKit/537.36 Chrome/120 Safari/537.36"
FILES={3031241:("TEP_FaultFree_Training.RData",24678017,"ec126484534331f85001d8c4ebce6d17"),
       3031240:("TEP_FaultFree_Testing.RData",47327663,"38ad9810fc871026157086ae2c2f0ee9"),
       3031242:("TEP_Faulty_Training.RData",494063194,"c5f594d54c47e620ff877feb58407fda"),
       3031243:("TEP_Faulty_Testing.RData",836882037,"556bdb64c83021bc0c5f92e427753565")}
def md5(p):
    h=hashlib.md5()
    with open(p,"rb") as f:
        for c in iter(lambda:f.read(1<<20),b""): h.update(c)
    return h.hexdigest()
log=[]
for fid,(name,size,want) in FILES.items():
    dst=os.path.join(RAW,name)
    if os.path.exists(dst) and os.path.getsize(dst)==size and md5(dst)==want:
        print(f"OK cache {name}",flush=True); log.append(dict(arquivo=name,bytes=size,md5=want,status="ok-cache")); continue
    for tent in range(1,6):
        t=time.time(); print(f"baixando {name} ({size/1e6:.0f} MB) tent {tent}",flush=True)
        r=subprocess.run(["curl","-sS","-L","-A",UA,"--retry","3","-C","-","-o",dst,
                          f"https://dataverse.harvard.edu/api/access/datafile/{fid}"])
        if r.returncode==0 and os.path.exists(dst) and os.path.getsize(dst)==size:
            got=md5(dst); dt=time.time()-t
            if got==want: print(f"  OK {dt:.0f}s md5 confere",flush=True); log.append(dict(arquivo=name,bytes=size,md5=got,status=f"ok {dt:.0f}s")); break
            print(f"  MD5 diverge {got}",flush=True)
        else: print(f"  rc={r.returncode} size={os.path.getsize(dst) if os.path.exists(dst) else 0}",flush=True)
        if os.path.exists(dst): os.remove(dst)
        time.sleep(10)
    else: print(f"FALHA {name}",flush=True); sys.exit(1)
json.dump(log,open(os.path.join(META,"download_manifest.json"),"w"),indent=2)
print("DOWNLOAD COMPLETO",flush=True)


In [ ]:
!python3 -u src/01_download.py

### 2. Conversão para Parquet com memória constante
Ler o arquivo de 837 MB com `pyreadr` materializa 9,6 M linhas em R **e** em pandas ao mesmo tempo (> 8 GB) — estourou a memória do Colab (12 GB) e estouraria a de um notebook de 8 GB. Solução: o R grava **uma parte por classe** (`saveRDS`); o Python lê parte a parte e anexa a um único Parquet em `float32` (precisão de ~7 dígitos, suficiente para sensores; metade da memória).

In [ ]:
%%writefile /content/ProjetoA_WP1A/src/convert_rdata.py
"""convert_rdata.py — converte .RData → Parquet float32 com MEMÓRIA CONSTANTE.
R nativo carrega o objeto e grava uma parte .rds por classe (faultNumber); Python lê cada parte com
pyreadr e anexa a um único Parquet (pyarrow ParquetWriter). Evita materializar 9,6 M linhas duas vezes."""
import os, sys, glob, subprocess, shutil, time, numpy as np, pyreadr, pyarrow as pa, pyarrow.parquet as pq
ROOT=os.environ.get("WP1A_ROOT") or os.path.join(os.path.dirname(os.path.abspath(__file__)),"..")
RAW=os.path.join(ROOT,"data","raw"); PROC=os.path.join(ROOT,"data","processed"); os.makedirs(PROC,exist_ok=True)
RS='args<-commandArgs(TRUE); nm<-load(args[1]); df<-get(nm[1]); cat("objeto:",nm[1],"dims:",dim(df),"\n"); ' \
   'for(k in sort(unique(df$faultNumber))){ saveRDS(df[df$faultNumber==k,], file.path(args[2], sprintf("part_%02d.rds",k)), compress=FALSE) }'
ID=["faultNumber","simulationRun","sample"]
for name in ["TEP_FaultFree_Training","TEP_FaultFree_Testing","TEP_Faulty_Training","TEP_Faulty_Testing"]:
    pq_path=os.path.join(PROC,name+".parquet")
    if os.path.exists(pq_path): print("parquet existe",name,flush=True); continue
    src=os.path.join(RAW,name+".RData"); parts=os.path.join(PROC,"_parts_"+name); shutil.rmtree(parts,ignore_errors=True); os.makedirs(parts)
    t=time.time(); r=subprocess.run(["Rscript","-e",RS,src,parts],capture_output=True,text=True); print(name,r.stdout.strip(),r.stderr.strip()[-300:],flush=True)
    if r.returncode!=0: sys.exit(f"Rscript falhou em {name}")
    writer=None; n=0
    for part in sorted(glob.glob(os.path.join(parts,"part_*.rds"))):
        df=list(pyreadr.read_r(part).values())[0]
        for c in df.columns: df[c]=df[c].astype("int32" if c in ID else "float32")
        tb=pa.Table.from_pandas(df,preserve_index=False)
        if writer is None: writer=pq.ParquetWriter(pq_path,tb.schema,compression="snappy")
        writer.write_table(tb); n+=len(df); del df,tb; os.remove(part)
    writer.close(); shutil.rmtree(parts,ignore_errors=True)
    print(f"  → {name}.parquet {n:,} linhas em {time.time()-t:.0f}s",flush=True)
print("CONVERSAO COMPLETA",flush=True)


In [ ]:
!python3 -u src/convert_rdata.py

### 3. Auditoria da base (WP1A §9.1)
Responde, por lotes, aos 8 itens exigidos: linhas/colunas, nomes das variáveis, classes, execuções por classe, amostras por execução, nulos/infinitos, duplicatas, relação entre arquivos. Gera a **Tabela 1** do artigo.

In [ ]:
%%writefile /content/ProjetoA_WP1A/src/02_auditoria.py
"""02_auditoria.py — Etapa 1 do WP1A (§9.1): auditoria da base A PARTIR DOS PARQUETS, em lotes (memória constante)."""
import os, json, numpy as np, pandas as pd, pyarrow.parquet as pq, pyarrow.compute as pc
ROOT=os.environ.get("WP1A_ROOT") or os.path.join(os.path.dirname(os.path.abspath(__file__)),"..")
RAW=os.path.join(ROOT,"data","raw"); PROC=os.path.join(ROOT,"data","processed")
TAB=os.path.join(ROOT,"results","tables"); META=os.path.join(ROOT,"results","metadata"); REP=os.path.join(ROOT,"reports")
FILES=["TEP_FaultFree_Training","TEP_FaultFree_Testing","TEP_Faulty_Training","TEP_Faulty_Testing"]; ID=["faultNumber","simulationRun","sample"]
aud={}; rows=[]
for name in FILES:
    pf=pq.ParquetFile(os.path.join(PROC,name+".parquet")); cols=pf.schema_arrow.names; xcols=[c for c in cols if c not in ID]
    nulos=0; infs=0; ids=[]
    for b in pf.iter_batches(batch_size=500_000):
        for c in xcols: nulos+=b.column(c).null_count
        X=np.column_stack([b.column(c).to_numpy(zero_copy_only=False) for c in xcols]); infs+=int(np.isinf(X).sum()); nulos+=int(np.isnan(X).sum())
        ids.append(b.select(ID).to_pandas())
    ids=pd.concat(ids,ignore_index=True); g=ids.groupby(ID[:2]).size()
    a=dict(arquivo=name,linhas=int(len(ids)),colunas=len(cols),n_variaveis_processo=len(xcols),variaveis=xcols,
           classes=sorted(int(x) for x in ids.faultNumber.unique()),n_classes=int(ids.faultNumber.nunique()),
           runs_por_classe={int(k):int(v) for k,v in ids.groupby("faultNumber").simulationRun.nunique().items()},
           amostras_por_run=dict(min=int(g.min()),max=int(g.max())),sample_min=int(ids["sample"].min()),sample_max=int(ids["sample"].max()),
           run_min=int(ids.simulationRun.min()),run_max=int(ids.simulationRun.max()),nulos_ou_nan=int(nulos),infinitos=int(infs),
           duplicatas_chave=int(ids.duplicated(subset=ID).sum()),
           rdata_bytes=os.path.getsize(os.path.join(RAW,name+".RData")) if os.path.exists(os.path.join(RAW,name+".RData")) else None,
           parquet_bytes=os.path.getsize(os.path.join(PROC,name+".parquet")))
    aud[name]=a; rp=set(a["runs_por_classe"].values())
    rows.append(dict(Arquivo=name,Linhas=a["linhas"],Colunas=a["colunas"],Classes=a["n_classes"],Runs_por_classe=(rp.pop() if len(rp)==1 else "varia"),
                     Amostras_por_run=(a["amostras_por_run"]["min"] if a["amostras_por_run"]["min"]==a["amostras_por_run"]["max"] else "varia"),
                     Nulos=a["nulos_ou_nan"],Infinitos=a["infinitos"],Duplicatas=a["duplicatas_chave"])); print(name,a["linhas"],a["n_classes"],flush=True)
v=aud["TEP_Faulty_Training"]["variaveis"]
aud["_consistencia"]=dict(mesmas_variaveis_treino_teste=(v==aud["TEP_Faulty_Testing"]["variaveis"]),n_xmeas=sum(c.startswith("xmeas") for c in v),n_xmv=sum(c.startswith("xmv") for c in v),
    sementes_treino_teste_nao_sobrepostas="por construção do dataset (Rieth et al., 2017); arquivos distintos")
json.dump(aud,open(os.path.join(META,"auditoria.json"),"w"),indent=2,default=str); pd.DataFrame(rows).to_csv(os.path.join(TAB,"tab01_caracterizacao_base.csv"),index=False)
c=aud["_consistencia"]
with open(os.path.join(REP,"01-auditoria.md"),"w") as f:
    f.write("# Relatório de auditoria da base — WP1A Etapa 1 (§9.1)\n\nFonte: Rieth et al. (2017), Harvard Dataverse, DOI 10.7910/DVN/6C3JR1. MD5 de cada .RData verificado no download.\n\n## Tabela 1 — Caracterização da base\n\n"+pd.DataFrame(rows).to_markdown(index=False)+"\n\n")
    f.write(f"## Variáveis de processo\n\n- Total: **{aud['TEP_Faulty_Training']['n_variaveis_processo']}** (XMEAS: {c['n_xmeas']} · XMV: {c['n_xmv']})\n- Mesmas variáveis em treino e teste: **{c['mesmas_variaveis_treino_teste']}**\n\n## Verificações por arquivo\n\n")
    for name in FILES:
        a=aud[name]; f.write(f"### {name}\n- classes: {a['classes']}\n- runs por classe: {sorted(set(a['runs_por_classe'].values()))} · simulationRun {a['run_min']}–{a['run_max']}\n- amostras por run: {a['amostras_por_run']['min']}–{a['amostras_por_run']['max']} · sample {a['sample_min']}–{a['sample_max']}\n- nulos/NaN {a['nulos_ou_nan']} · infinitos {a['infinitos']} · duplicatas de chave {a['duplicatas_chave']}\n- .RData {(a['rdata_bytes'] or 0)/1e6:.0f} MB → parquet float32 {a['parquet_bytes']/1e6:.0f} MB\n\n")
print("AUDITORIA COMPLETA",flush=True)


In [ ]:
!python3 -u src/02_auditoria.py && cat reports/01-auditoria.md

### 4. Divisão por execução e manifesto (WP1A §9.3 · entrega A3)
A **execução completa** (`simulationRun`) é a unidade indivisível: um run vai inteiro para treino, validação ou teste. Treino = 100 runs/classe, validação = 50 (ambos do arquivo *Training*), teste = todos os 500 do *Testing*. A interseção entre conjuntos é verificada programaticamente (por arquivo de origem — o mesmo número de run em *Training* e *Testing* são simulações distintas). O manifesto gerado aqui é **idêntico byte a byte** ao gerado no Mac (mesma semente).

In [ ]:
%%writefile /content/ProjetoA_WP1A/src/03_split.py
"""03_split.py — Etapa 3 do WP1A (§9.3): divisão experimental POR RUN.
Gera o manifesto de divisão (entrega A3) e verifica interseção vazia entre conjuntos.
Configuração: treino 100 runs/classe · validação 50 runs/classe (ambos do Training)
              teste = TODOS os 500 runs/classe do Testing (inferência é barata).
Precedente: Koçak et al. (2026) usaram 200/500; Lyu et al. (2026) usaram 100/50/200."""
import os, json, numpy as np, pandas as pd
ROOT=os.environ.get("WP1A_ROOT") or os.path.join(os.path.dirname(os.path.abspath(__file__)),"..")
PROC=os.path.join(ROOT,"data","processed"); META=os.path.join(ROOT,"results","metadata"); CFG=os.path.join(ROOT,"configs")
SEED=42; N_TREINO=100; N_VAL=50
rng=np.random.RandomState(SEED)
tr_ff=pd.read_parquet(os.path.join(PROC,"TEP_FaultFree_Training.parquet"),columns=["faultNumber","simulationRun"]).drop_duplicates()
tr_f =pd.read_parquet(os.path.join(PROC,"TEP_Faulty_Training.parquet"),columns=["faultNumber","simulationRun"]).drop_duplicates()
te_ff=pd.read_parquet(os.path.join(PROC,"TEP_FaultFree_Testing.parquet"),columns=["faultNumber","simulationRun"]).drop_duplicates()
te_f =pd.read_parquet(os.path.join(PROC,"TEP_Faulty_Testing.parquet"),columns=["faultNumber","simulationRun"]).drop_duplicates()
training=pd.concat([tr_ff,tr_f]); testing=pd.concat([te_ff,te_f])
rows=[]  # (origem, classe, run, conjunto) — origem distingue Training de Testing (sementes distintas por construção)
for cls,g in training.groupby("faultNumber"):
    runs=g.simulationRun.to_numpy().copy(); rng.shuffle(runs)  # .copy(): pandas>=3 retorna array read-only
    for r in runs[:N_TREINO]: rows.append(("Training",int(cls),int(r),"treino"))
    for r in runs[N_TREINO:N_TREINO+N_VAL]: rows.append(("Training",int(cls),int(r),"validacao"))
    for r in runs[N_TREINO+N_VAL:]: rows.append(("Training",int(cls),int(r),"nao_usado_training"))
for cls,g in testing.groupby("faultNumber"):
    for r in g.simulationRun: rows.append(("Testing",int(cls),int(r),"teste"))
man=pd.DataFrame(rows,columns=["origem","faultNumber","simulationRun","conjunto"])
# --- VERIFICAÇÃO OBRIGATÓRIA: nenhuma (classe,run) em mais de um conjunto usado
usados=man[man.conjunto.isin(["treino","validacao","teste"])]
dup=usados.duplicated(subset=["origem","faultNumber","simulationRun"]).sum()  # dentro do mesmo arquivo de origem
assert dup==0, f"VAZAMENTO: {dup} pares (classe,run) em mais de um conjunto"
res=man.groupby(["conjunto","faultNumber"]).size().unstack(0).fillna(0).astype(int)
man.to_csv(os.path.join(META,"manifesto_divisao.csv"),index=False)
json.dump(dict(seed=SEED,n_treino_por_classe=N_TREINO,n_validacao_por_classe=N_VAL,teste="todos os runs do Testing",
               unidade="simulationRun (execução completa)",verificacao_intersecao_vazia=bool(dup==0),
               totais=man.conjunto.value_counts().to_dict()),open(os.path.join(CFG,"split.json"),"w"),indent=2)
print(res); print(f"\ninterseção entre conjuntos: {dup} → {'OK' if dup==0 else 'FALHA'}"); print("MANIFESTO GRAVADO")


In [ ]:
!python3 -u src/03_split.py && md5sum results/metadata/manifesto_divisao.csv

### 5. Análise exploratória (WP1A §9.2 · entrega A2)
Sobre o treino apenas: descritivas, correlação, PCA e o afastamento de cada falha em relação ao regime normal (em desvios-padrão do normal). Observação: essa medida capta deslocamentos de **média**; falhas de variância (10, 14, 19) ficam subestimadas.

In [ ]:
%%writefile /content/ProjetoA_WP1A/src/05_eda.py
"""05_eda.py — Etapa 2 do WP1A (§9.2): análise exploratória (entrega A2).
Roda sobre o conjunto de TREINO do manifesto (nunca sobre o teste). Gera figuras e tabelas."""
import os, json, numpy as np, pandas as pd, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
ROOT=os.environ.get("WP1A_ROOT") or os.path.join(os.path.dirname(os.path.abspath(__file__)),"..")
PROC=os.path.join(ROOT,"data","processed"); META=os.path.join(ROOT,"results","metadata")
TAB=os.path.join(ROOT,"results","tables"); FIG=os.path.join(ROOT,"results","figures")
man=pd.read_csv(os.path.join(META,"manifesto_divisao.csv")); tr_runs=man[man.conjunto=="treino"]
def load(nome):
    df=pd.read_parquet(os.path.join(PROC,nome+".parquet"))
    return df.merge(tr_runs[["faultNumber","simulationRun"]],on=["faultNumber","simulationRun"])
tr=pd.concat([load("TEP_FaultFree_Training"),load("TEP_Faulty_Training")])
X_cols=[c for c in tr.columns if c not in ("faultNumber","simulationRun","sample")]
print(f"treino: {len(tr):,} linhas, {len(X_cols)} variáveis, {tr.faultNumber.nunique()} classes",flush=True)
# 1. estatísticas descritivas por variável
desc=tr[X_cols].describe().T; desc["cv"]=desc["std"]/desc["mean"].abs()
desc.round(4).to_csv(os.path.join(TAB,"tab_eda_descritivas.csv"))
# 2. distribuição das classes (linhas e runs)
dist=tr.groupby("faultNumber").agg(linhas=("sample","size"),runs=("simulationRun","nunique"))
dist.to_csv(os.path.join(TAB,"tab_eda_distribuicao_classes.csv"))
# 3. correlação
sc=StandardScaler().fit(tr[X_cols]); Z=sc.transform(tr[X_cols])
corr=np.corrcoef(Z,rowvar=False); pd.DataFrame(corr,index=X_cols,columns=X_cols).round(3).to_csv(os.path.join(TAB,"tab_eda_correlacao.csv"))
fig,ax=plt.subplots(figsize=(9,8)); im=ax.imshow(corr,cmap="RdBu_r",vmin=-1,vmax=1)
ax.set_xticks(range(len(X_cols))); ax.set_xticklabels(X_cols,rotation=90,fontsize=5); ax.set_yticks(range(len(X_cols))); ax.set_yticklabels(X_cols,fontsize=5)
plt.colorbar(im,ax=ax,fraction=0.03); ax.set_title("Correlação de Pearson entre as 52 variáveis (treino, padronizado)")
plt.tight_layout(); plt.savefig(os.path.join(FIG,"fig_eda_correlacao.png"),dpi=200); plt.close()
# 4. PCA
pca=PCA(n_components=10,random_state=42).fit(Z); ev=pca.explained_variance_ratio_
pd.DataFrame({"componente":range(1,11),"variancia_explicada":ev,"acumulada":np.cumsum(ev)}).round(4).to_csv(os.path.join(TAB,"tab_eda_pca_variancia.csv"),index=False)
# amostra p/ scatter: 2000 pontos por classe
idx=tr.groupby("faultNumber").sample(n=2000,random_state=42).index
P=pca.transform(Z[tr.index.get_indexer(idx)] if False else sc.transform(tr.loc[idx,X_cols]))
fig,ax=plt.subplots(figsize=(9,7)); y=tr.loc[idx,"faultNumber"].to_numpy()
for c in sorted(np.unique(y)):
    m=y==c; ax.scatter(P[m,0],P[m,1],s=2,alpha=0.35,label=f"{c}",color=("black" if c==0 else None))
ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)"); ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)"); ax.set_title("Projeção PCA das 21 classes (2.000 amostras/classe; 0 = normal em preto)")
ax.legend(markerscale=6,fontsize=6,ncol=3,title="faultNumber"); plt.tight_layout(); plt.savefig(os.path.join(FIG,"fig_eda_pca.png"),dpi=200); plt.close()
# 5. normal vs falha: distância padronizada média por variável, por falha
norm=tr[tr.faultNumber==0][X_cols].mean(); sd=tr[tr.faultNumber==0][X_cols].std()
sep=pd.DataFrame({f: ((tr[tr.faultNumber==f][X_cols].mean()-norm)/sd).abs() for f in range(1,21)}).T
sep.round(3).to_csv(os.path.join(TAB,"tab_eda_separacao_normal_vs_falha.csv"))
fig,ax=plt.subplots(figsize=(10,6)); im=ax.imshow(sep.to_numpy(),aspect="auto",cmap="viridis")
ax.set_yticks(range(20)); ax.set_yticklabels([f"IDV{f}" for f in range(1,21)],fontsize=7); ax.set_xticks(range(len(X_cols))); ax.set_xticklabels(X_cols,rotation=90,fontsize=5)
plt.colorbar(im,ax=ax,label="|média_falha − média_normal| / dp_normal"); ax.set_title("Afastamento de cada falha em relação ao regime normal, por variável")
plt.tight_layout(); plt.savefig(os.path.join(FIG,"fig_eda_separacao.png"),dpi=200); plt.close()
sumario=dict(linhas=int(len(tr)),variaveis=len(X_cols),classes=int(tr.faultNumber.nunique()),
             pca_var_2comp=float(np.cumsum(ev)[1]),pca_var_10comp=float(np.cumsum(ev)[9]),
             pares_corr_abs_maior_0_9=int(((np.abs(corr)>0.9).sum()-len(X_cols))//2),
             falhas_mais_proximas_do_normal=sep.mean(axis=1).nsmallest(5).round(3).to_dict(),
             falhas_mais_distantes_do_normal=sep.mean(axis=1).nlargest(5).round(3).to_dict())
json.dump(sumario,open(os.path.join(META,"eda_sumario.json"),"w"),indent=2)
print(json.dumps(sumario,indent=2)); print("EDA COMPLETA",flush=True)


In [ ]:
!python3 -u src/05_eda.py && cat results/metadata/eda_sumario.json

### Resultados obtidos nesta etapa (execução de 06/09/2026)
| Arquivo | Linhas | Classes | Runs/classe | Amostras/run | Nulos | Infinitos | Duplicatas |
|---|---|---|---|---|---|---|---|
| FaultFree_Training | 250.000 | 1 | 500 | 500 | 0 | 0 | 0 |
| FaultFree_Testing | 480.000 | 1 | 500 | 960 | 0 | 0 | 0 |
| Faulty_Training | 5.000.000 | 20 | 500 | 500 | 0 | 0 | 0 |
| Faulty_Testing | 9.600.000 | 20 | 500 | 960 | 0 | 0 | 0 |

52 variáveis = 41 `xmeas` + 11 `xmv`, idênticas em treino e teste. PCA: 2 componentes explicam 43,9%; 10 explicam 74,5%. 21 pares com |r| > 0,9. Falhas mais próximas do normal (média): 9, 14, 10, 15, 19; mais distantes: 6, 18, 2, 1.